# Breast Cancer Diagnosis Classification

**Internship project | Machine learning classification**

This notebook builds and evaluates a reproducible machine-learning model that classifies breast tumor measurements as **benign (B)** or **malignant (M)**. It expands the original exercise with data checks, exploratory visualizations, a baseline, leakage-safe preprocessing, cross-validation, and holdout evaluation.

> **Important:** This is an educational demonstration using a public dataset. It is not a medical device, has not been clinically validated, and must not be used to diagnose, screen, or guide treatment for any person.

## Project objectives

- Inspect and validate the dataset before modeling.
- Explore diagnosis balance and selected feature relationships.
- Compare a simple majority-class baseline with logistic regression.
- Evaluate performance on a stratified holdout set and with cross-validation.
- Review model behavior while documenting important limitations.

The target is `diagnosis` (`M` = malignant, `B` = benign). The input features are the 30 numeric nucleus measurements. Patient/sample identifiers and empty columns are excluded.

## 1. Imports and reproducibility

Dependencies: pandas, NumPy, Matplotlib, seaborn, and scikit-learn.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 2529
sns.set_theme(style="whitegrid", context="notebook")

## 2. Load the data

The notebook downloads the CSV from the YBI Foundation's public dataset repository. An internet connection is required. Review the source and its terms before reusing the dataset in other work.

In [ ]:
DATA_URL = "https://github.com/YBIFoundation/Dataset/raw/main/Cancer.csv"
cancer = pd.read_csv(DATA_URL)

print(f"Dataset shape: {cancer.shape[0]} rows x {cancer.shape[1]} columns")
display(cancer.head())

## 3. Validate and prepare features

Check the expected target, class labels, data types, and missing values. The `id` column is an identifier rather than a measurement, and columns that are completely empty are discarded. Any partially missing or non-numeric feature values are surfaced rather than silently ignored.

In [ ]:
if "diagnosis" not in cancer.columns:
    raise ValueError("The dataset must contain a 'diagnosis' target column.")
if cancer["diagnosis"].isna().any():
    raise ValueError("The diagnosis target contains missing values.")

class_labels = set(cancer["diagnosis"].unique())
if class_labels != {"B", "M"}:
    raise ValueError(f"Expected diagnosis labels 'B' and 'M'; got {class_labels}.")

y = cancer["diagnosis"].copy()
X = cancer.drop(columns=["diagnosis", "id"], errors="ignore")
X = X.dropna(axis=1, how="all")

non_numeric_columns = X.select_dtypes(exclude=np.number).columns.tolist()
if non_numeric_columns:
    raise TypeError(f"Expected numeric predictors; found {non_numeric_columns}.")
if X.isna().any().any():
    missing = X.columns[X.isna().any()].tolist()
    raise ValueError(f"Predictor columns contain missing values: {missing}.")

print(f"Predictors: {X.shape[1]}")
print("Target distribution:")
display(y.value_counts().rename_axis("diagnosis").to_frame("count"))
print("Missing values by column:")
display(cancer.isna().sum().sort_values(ascending=False).head(10).to_frame("missing_count"))
display(X.describe().T)

## 4. Exploratory data analysis

These plots provide a quick view of class balance and selected measurement patterns. Visual separation in this dataset does not establish clinical usefulness or causal relationships.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.countplot(data=cancer, x="diagnosis", order=["B", "M"], ax=axes[0])
axes[0].set(title="Diagnosis class counts", xlabel="Diagnosis (B = benign, M = malignant)", ylabel="Samples")

for label, color in [("B", "#4C72B0"), ("M", "#DD8452")]:
    sns.histplot(
        data=cancer.loc[cancer["diagnosis"] == label],
        x="radius_mean",
        stat="density",
        common_norm=False,
        element="step",
        fill=False,
        label=label,
        color=color,
        ax=axes[1],
    )
axes[1].set(title="Mean radius by diagnosis", xlabel="Mean radius", ylabel="Density")
axes[1].legend(title="Diagnosis")
plt.tight_layout()
plt.show()

In [ ]:
mean_features = [column for column in X.columns if column.endswith("_mean")]
correlation = cancer[mean_features].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(correlation, cmap="coolwarm", center=0, square=True, linewidths=0.3)
plt.title("Correlation between mean nucleus measurements")
plt.tight_layout()
plt.show()

## 5. Stratified train/test split

The test set is held out until final evaluation. Stratification preserves the approximate diagnosis proportions in both splits. Scaling is placed inside the model pipeline so each cross-validation fold learns scaling parameters from its training portion only.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print("Training diagnosis proportions:")
display(y_train.value_counts(normalize=True).rename("proportion").to_frame())
print("Test diagnosis proportions:")
display(y_test.value_counts(normalize=True).rename("proportion").to_frame())

## 6. Baseline and logistic regression model

The baseline always predicts the most common training class. The logistic-regression pipeline standardizes features before fitting; `max_iter` is raised to support convergence. Model choice and settings here are illustrative, not clinically optimized.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
    ]
)
model.fit(X_train, y_train)

print(f"Baseline test accuracy: {accuracy_score(y_test, baseline_predictions):.3f}")
print("The fitted logistic-regression pipeline is ready for validation.")

## 7. Cross-validation on the training set

Stratified five-fold cross-validation estimates training-set performance across multiple splits. The held-out test data is not used in this step. ROC-AUC treats malignant (`M`) as the positive class.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "precision_M": "precision_macro",
    "recall_M": "recall_macro",
    "f1_M": "f1_macro",
    "roc_auc": "roc_auc",
}

cv_results = cross_validate(
    model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
)

cv_summary = pd.DataFrame(
    {
        "mean": {metric: cv_results[f"test_{metric}"].mean() for metric in scoring},
        "std": {metric: cv_results[f"test_{metric}"].std() for metric in scoring},
    }
)
display(cv_summary.round(3))

## 8. Final holdout evaluation

Report several metrics rather than accuracy alone. In particular, malignant-class recall measures the fraction of malignant samples correctly identified in this test split. Neither this metric nor any other result should be interpreted as evidence of clinical safety.

In [ ]:
y_pred = model.predict(X_test)
malignant_index = list(model.classes_).index("M")
y_score_malignant = model.predict_proba(X_test)[:, malignant_index]

holdout_metrics = pd.Series(
    {
        "accuracy": accuracy_score(y_test, y_pred),
        "malignant_precision": precision_score(y_test, y_pred, pos_label="M"),
        "malignant_recall": recall_score(y_test, y_pred, pos_label="M"),
        "malignant_f1": f1_score(y_test, y_pred, pos_label="M"),
        "roc_auc_malignant_positive": roc_auc_score(y_test == "M", y_score_malignant),
    },
    name="holdout score",
)
display(holdout_metrics.to_frame().round(3))

print(classification_report(y_test, y_pred, labels=["B", "M"], target_names=["Benign (B)", "Malignant (M)"], zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=["B", "M"],
    display_labels=["Benign (B)", "Malignant (M)"],
    cmap="Blues",
    ax=axes[0],
    colorbar=False,
)
axes[0].set_title("Holdout confusion matrix")

RocCurveDisplay.from_predictions(
    y_test == "M",
    y_score_malignant,
    name="Logistic regression",
    ax=axes[1],
)
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
axes[1].set_title("Holdout ROC curve (M is positive)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 9. Inspect feature importance

Permutation importance measures the change in holdout ROC-AUC after shuffling one feature at a time. It is a model-specific exploratory summary, can be affected by correlated predictors, and is not a causal explanation. Because the test set is being inspected here, do not use these results to repeatedly tune and re-report performance on the same holdout set.

In [ ]:
importance = permutation_importance(
    model,
    X_test,
    y_test == "M",
    scoring="roc_auc",
    n_repeats=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance_summary = pd.DataFrame(
    {
        "feature": X.columns,
        "mean_auc_decrease": importance.importances_mean,
        "std": importance.importances_std,
    }
).sort_values("mean_auc_decrease", ascending=False)

display(importance_summary.head(10).round(4))
top_features = importance_summary.head(10).sort_values("mean_auc_decrease")
plt.figure(figsize=(9, 6))
plt.barh(top_features["feature"], top_features["mean_auc_decrease"], xerr=top_features["std"])
plt.xlabel("Mean decrease in holdout ROC-AUC after shuffling")
plt.title("Top 10 permutation importances")
plt.tight_layout()
plt.show()

## 10. Conclusions and limitations

Use the metrics and plots produced above to write a short conclusion. Compare the model with the majority-class baseline and describe the trade-offs among malignant precision, malignant recall, and false positives/negatives. Avoid relying on one score or implying that this experiment establishes readiness for real-world use.

### Limitations

- This is a small, pre-existing dataset; its samples, collection process, and class distribution may not represent other populations or clinical settings.
- A single random holdout split and cross-validation do not substitute for external validation on independent, appropriately collected data.
- Feature correlations, measurement differences, dataset shift, and possible subgroup performance differences can affect results.
- Classification thresholds and the costs of false negatives and false positives require clinical context; this notebook does not establish them.
- The model has not undergone clinical validation, regulatory review, or prospective testing. It must not be used for patient care.

### Suggested next steps for an academic project

Document the dataset version and environment, use nested validation if model selection is performed, predefine evaluation criteria, investigate subgroup performance where data permits, and seek independent expert review. Any clinical use would require a separate, rigorous validation and governance process.

## References

- YBI Foundation, [Cancer dataset](https://github.com/YBIFoundation/Dataset/raw/main/Cancer.csv).
- Wolberg, W. H., Street, W. N., & Mangasarian, O. L. Breast Cancer Wisconsin (Diagnostic) data; available through the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic).
- scikit-learn documentation: [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html), and [Model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html).

---

**Project scope:** Demonstration of a supervised classification workflow for learning purposes only; not a diagnostic or clinical decision-support system.